# Реальная лабораторная 06 — Adult и Credit Default

Социально-экономические данные, аудит групповых срезов и ограничения выводов.

> **Как работать:** выполняй ячейки последовательно. Сначала сформулируй ожидание, затем запускай код и объясняй результат. Не удаляй строки и не исправляй значения автоматически до выяснения их смысла.

**Данные:** скачиваются из UCI Machine Learning Repository через `download_uci_datasets.py`. Первый запуск требует интернета; после сохранения CSV лабораторная выполняется локально.

## 0. Исследовательский контракт

Запиши единицу наблюдения, целевую аудиторию анализа, момент принятия решения и то, какие выводы допустимы. Разделяй описательное исследование, прогноз и причинные утверждения.

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

SEED = 42
RNG = np.random.default_rng(SEED)
DATA_DIR = Path("data/raw")
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 60)

def load_real(slug):
    path = DATA_DIR / f"{slug}.csv"
    if not path.exists():
        raise FileNotFoundError(
            f"Не найден {path}. Сначала выполните: "
            f"python download_uci_datasets.py --only {slug}"
        )
    return pd.read_csv(path, low_memory=False)

def section(title, text):
    display(Markdown(f"### {title}\n\n{text}"))

## Часть A. Adult — аудит социально-экономических срезов

In [ ]:
adult = load_real("adult")
print(adult.shape); display(adult.head()); display(adult.dtypes.to_frame("dtype"))
# UCI files sometimes encode unknown categorical values as '?' rather than NaN.
adult = adult.replace(r"^\s*\?\s*$", np.nan, regex=True)
target_a = next((c for c in adult.columns if c.lower() in {"income","class","target"}), None)
print("Target candidate:", target_a)
if target_a: display(adult[target_a].value_counts(dropna=False, normalize=True))

### Пропуски по группам

In [ ]:
for col in ["workclass","occupation","native-country"]:
    actual = next((c for c in adult.columns if c.lower()==col), None)
    if actual:
        display(adult.groupby(actual, dropna=False).size().sort_values(ascending=False).head(15).to_frame("rows"))
for col in ["sex","race"]:
    actual = next((c for c in adult.columns if c.lower()==col), None)
    if actual and target_a:
        display(pd.crosstab(adult[actual], adult[target_a], normalize="index").round(3))

## Часть B. Default of Credit Card Clients — оценка риска и fairness audit

In [ ]:
credit = load_real("default_credit_card")
print(credit.shape); display(credit.head()); display(credit.dtypes.to_frame("dtype"))
target_c = next((c for c in credit.columns if "default" in c.lower() or c.lower() in {"y","target"}), None)
print("Target candidate:", target_c)
if target_c: display(credit[target_c].value_counts(normalize=True).to_frame("share"))

### Временная семантика платежной истории

In [ ]:
pay_cols=[c for c in credit.columns if c.upper().startswith("PAY_")]
bill_cols=[c for c in credit.columns if c.upper().startswith("BILL_AMT") or c.upper().startswith("PAY_AMT")]
print("Payment status fields:", pay_cols)
print("Bill/payment amount fields:", bill_cols)
if pay_cols:
    display(credit[pay_cols].describe().T)
print("Сверь документацию: какие месяцы относятся к прошлому, а какой период — к target.")

## Методологические ограничения

Это учебная практика аудита групповых различий и смещений, а не инструкция для принятия решений о трудоустройстве, доходе или выдаче кредита. Групповая разница не доказывает дискриминацию и не устанавливает причинность; отсутствие разницы не доказывает справедливость. Проверяй состав выборки, базовые частоты, качество меток и последствия ошибок.

## Самостоятельные задания

1. Для Adult сравни доли пропусков по occupation и workclass; проверь, различаются ли они между группами.
2. Рассчитай базовые частоты target по race и sex, но явно укажи, почему эти показатели нельзя интерпретировать причинно.
3. Для credit dataset составь временную карту доступности PAY_*, BILL_AMT* и PAY_AMT*.
4. Раздели аудит на quality, performance-by-group и potential-impact; какие данные для настоящего fairness audit отсутствуют?
5. Напиши ограничения: исторический контекст, региональная применимость, смещение выборки, качество и допустимость использования чувствительных признаков.

## Итоговый отчёт

В финальной версии notebook должны быть: **вопрос → единица анализа → происхождение и ограничения данных → проверка качества → описательная статистика → закономерности → неопределённость → риски смещения/утечки → выводы → следующие проверки**. Каждый вывод должен ссылаться на таблицу, график или расчёт.